In [ ]:
### GPT Layer : LayerNorm layer 모듈

class LayerNorm(nn.Module):
    def __init__(self, d_embedding):
        super().__init__()

        ## 정규화를 위한 scale, shfit 상수 값 계산
        self.eps = 1e-5
        self.scale = nn.Parameter( torch.ones(d_embedding) )
        self.shift = nn.Parameter( torch.zeros(d_embedding) )

    def forward(self, x)
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        norm_x = (x - mean) / torch.sqrt(var + self.eps)

        return self.scale * norm_x + self.shfit

In [ ]:
### GPT Layer : GELU layer

class GELU(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x):
        return 0.5 * x * (1 + torch.tanh( torch.sqrt(torch.tensor(2.0 / torch.pi)) * (x + 0.044715 * torch.pow(x, 3)) ))

In [ ]:
### GPT Layer : FeedForward layer

class FeedForward(nn.Module):
    def __init__(self, cfg):        # cfg["emb_dim"], 4배
        super().__init__()

        ## 1. 차원 확장 -> GELU -> 차원 압축
        self.layers = nn.Sequential(
            nn.Linear(cfg["emb_dim"], 4 * cfg["emb_dim"]),
            GELU(),
            nn.Linear(4 * cfg["emb_dim"], cfg["emb_dim"]),
        )

    def forward(self, x):    
        return self.layers(x)

In [ ]:
### GPT Layer : TransformerBlock layer
## LyarNorm -> Attention -> Add(Residual) -> LayerNorm -> FeedForward -> Add(Residual)

class TransformerBlock(nn.Module):  
    def __init__(self, cfg):
        super().__init__()

        ## 1. attn, ff, norm1, norm2, drop_shortcut 정의
        self.attn = MultiHeadAttention(
            d_in = cfg["d_emb"],
            d_out = cfg["d_emb"],
            context_length = cfg["context_length"],
            num_heads = cfg["n_heads"],
            dropout = cfg["dropout"],
            qkv_bias = cfg["qkv_bias"]
        )
        self.ff = FeedForward(cfg)
        self.norm1 = LayerNorm(cfg["d_emb"])
        self.norm2 = LayerNorm(cfg["d_emb"])
        self.drop_shortcut = nn.Dropout(cfg["dropout"])

    def forward(self, x):
        ### 1. Attention Block

        ## 1-1. shortcut 저장
        shortcut = x

        ## 1-2. LayerNorm
        x = self.norm1(x)

        ## 1-3. Attention
        x = self.attn(x)

        ## 1-4. Dropout + Add(Residual)
        x = self.drop_shortcut(x)
        x = x + shortcut


        ### 2. FeedForward Block

        ## 2-1. shortcut 저장
        shortcut = x
        
        ## 2-2. LayerNorm
        x = self.norm2(x)
        
        ## 2-3. FeedForward
        x = self.ff(x)

        ## 2-4. Dropout + Add(Residual)
        x = self.drop_shortcut(x)
        x = x + shortcut

        return x

In [ ]:
### GPT Model
## Embedding -> Transformer Blocks -> Final Norm -> Output Head

class GPTModel(nn.Module):  
    def __init__(self, cfg):
        super().__init__()

        ## 1. embeding layer 정의 (token, pos)
        self.tok_emb = nn.Embedding( cfg["vocab_size"], cfg["emb_dim"] )
        self.pos_emb = nn.Embedding( cfg["context_length"], cfg["emb_dim"] )

        ## 2. dropout layer
        self.drop_emb = nn.Dropout( cfg["dropout"] )

        ## 3. transformer block 정의
        self.trf_blocks = nn.Sequential(
            *[ TransformerBlock(cfg) for _ in range( cfg["n_layers"] ) ]
        )

        ## 4. firnal norm layer
        self.final_norm = LayerNorm( cfg["emb_dim"] )
        
        ## 5. out head layer
        self.out_head = nn.Linear( cfg["emb_dim"], cfg["vocab_size"], bias=False )


    def forward(self, in_idx):
        ## 1. 상수 정의
        batch_size, seq_len = in_idx.shape      # [B, N]

        ## 2. input embedding 생성
        tok_embeds = self.tok_emb(in_idx)
        pos_embeds = self.pos_emb( torch.arange(seq_len, device=in_idx.device) )
        x = tok_embeds + pos_embeds

        ## 3. dropout
        x = self.drop_emb( x )

        ## 4. transformer 통과
        x = self.trf_blocks( x )

        ## 5. 최종 norm
        x = self.final_norm( x )

        ## 6. 각 단어(토큰)에 대한 예측 점수(logits) 계산
        logits = self.out_head( x )

        return logits

In [ ]:
### Utility Function : generate_text_simple()

def generate_text_simple( model, idx, max_new_tokens, context_size ):   # idx : 토큰 index 목록 -> GPTModel 에서 실제 embdding 적용
    """
        현재 문맥을 넣어 다음 텍스트를 max_new_tokens 만큼 생성, (최대 길이는 context_size)
        idx: 현재 문맥의 토큰 인덱스들 [B, Timestep]
    """
    for _ in range(max_new_tokens):
        ## 1. 최대 지원 길이로 제한
        idx_cond = idx[:, -context_size:]

        ## 2. 모델 예측
        with torch.no_grad():
            logits = model( idx_cond )

        ## 3. 마지막 타임스텝의 예측값만 가져옴 
        logits = logits[:, -1, :]       # [B, N, vocab_size] -> [B, vocab_size]

        ## 4. 가장 확률(로짓값)이 높은 토큰 선택 (greedy)
        idx_next = torch.argmax( logits, dim=-1, keepdim=True )

        ## 5. 예측한 토큰을 이어 붙임
        idx = torch.cat( (idx, idx_next), dim=1 )

    ## 6. 최종 idx 반환
    return idx


In [ ]:
### GPTModel 사용 예제

def main():
    # GPT-2 Small 모델 설정값
    GPT_CONFIG_124M = {
        "vocab_size": 50257,     # 단어 집합 크기
        "context_length": 1024,  # 최대 문맥 길이
        "emb_dim": 768,          # 임베딩 차원
        "n_heads": 12,           # 어텐션 헤드 수
        "n_layers": 12,          # 레이어 수
        "drop_rate": 0.1,        # 드롭아웃 비율
        "qkv_bias": False        # Q,K,V 편향 사용 여부
    }

    forch.manual_seed(123)

    ## 1. 모델 초기화 및 평가 모드 설정
    model = GPTModel( GPT_CONFIG_124M )
    model.eval()

    start_context = "Hello, I am"

    ## 2. 입력 텍스트 인코딩
    tokenizer = tiktoken.get_encoding("gpt2")
    encoded = tokenizer.encode( start_context )
    encoded_tensor = torch.tensor( encoded ).unsqueeze( 0 )     # 배치 차원 추가

    ## 3. 텍스트 생성 실행
    out = generate_text_simple(
        model = model,
        idx_cond = encoded_tensor,
        max_new_tokens = 10,
        context_size = GPT_CONFIG_124M["context_length] )

    ## 4. 생성된 결과 디코딩
    decoded_text = tokenizer.decode( out.squeeze( 0 ).tolist() )     # 배치 차원 제거 후 tonsor -> list

    print( decoded_text )


if __name__ == "__main__":
    main()